In [ ]:
source_language_code = input("Enter the 2-letter ISO 639-1 code for the source language (e.g., 'en' for English, 'es' for Spanish): ")
target_language_code = input("Enter the 2-letter ISO 639-1 code for the target language (e.g., 'en' for English, 'es' for Spanish): ")

print(f"Source Language Code: {source_language_code}")
print(f"Target Language Code: {target_language_code}")

Enter the 2-letter ISO 639-1 code for the source language (e.g., 'en' for English, 'es' for Spanish): pa
Enter the 2-letter ISO 639-1 code for the target language (e.g., 'en' for English, 'es' for Spanish): pa
Source Language Code: pa
Target Language Code: pa


In [ ]:
from google.colab import files
from pydub import AudioSegment
import os

print("Please upload your audio file (e.g., .wav, .mp3):")

uploaded = files.upload()

# Assuming only one file is uploaded for simplicity
file_name = next(iter(uploaded))
base_name, ext = os.path.splitext(file_name)

print(f"Uploaded file: {file_name}")

wav_file_path = ''

if ext.lower() == '.wav':
    print("File is already in WAV format.")
    wav_file_path = file_name
    # Save the uploaded WAV file to disk
    with open(wav_file_path, 'wb') as f:
        f.write(uploaded[file_name])
elif ext.lower() in ['.mp3', '.m4a', '.flac', '.ogg']:
    print(f"Converting {ext} to WAV format...")
    # pydub requires the file to be saved to disk first to determine its format
    temp_original_path = file_name
    with open(temp_original_path, 'wb') as f:
        f.write(uploaded[file_name])

    audio = AudioSegment.from_file(temp_original_path, format=ext.lstrip('.'))
    wav_file_path = base_name + '.wav'
    audio.export(wav_file_path, format="wav")
    print(f"Conversion complete. Original file removed, converted WAV saved as: {wav_file_path}")
    # Clean up the original uploaded file if it was converted
    os.remove(temp_original_path)
else:
    print(f"Unsupported audio format: {ext}. Please upload a .wav, .mp3, .m4a, .flac, or .ogg file.")
    wav_file_path = ""

print(f"Audio file ready for speech recognition at: {wav_file_path}")

Please upload your audio file (e.g., .wav, .mp3):


/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


Saving audio3.mp3 to audio3.mp3
Uploaded file: audio3.mp3
Converting .mp3 to WAV format...
Conversion complete. Original file removed, converted WAV saved as: audio3.wav
Audio file ready for speech recognition at: audio3.wav


In [ ]:
import speech_recognition as sr
import os

r = sr.Recognizer()

try:
    # Check if the WAV file exists before trying to open it
    if not os.path.exists(wav_file_path):
        raise FileNotFoundError(f"Audio file '{wav_file_path}' not found. Please ensure the audio file is uploaded and converted to WAV format.")

    # Open the WAV file
    with sr.AudioFile(wav_file_path) as source:
        print(f"Processing audio file: {wav_file_path}")
        audio_data = r.record(source)  # read the entire audio file

    # Use Google's Web Speech API to recognize the audio with the specified source language
    recognized_text = r.recognize_google(audio_data, language=source_language_code)
    print("Speech recognition complete.")
    print(f"Recognized Text ({source_language_code}): {recognized_text}")
except FileNotFoundError as e:
    recognized_text = "<Error: Audio file not found>"
    print(e)
    print("Please ensure you have run the 'Refine Audio Processing' cell and successfully uploaded/converted your audio file.")
except sr.UnknownValueError:
    recognized_text = "<Could not understand audio>"
    print(f"Google Web Speech API could not understand audio in {source_language_code}")
except sr.RequestError as e:
    recognized_text = "<Error with the speech recognition service>"
    print(f"Could not request results from Google Web Speech API service; {e}")

user_text_input = recognized_text
print(f"Recognized text stored in 'user_text_input': {user_text_input}")

Processing audio file: audio3.wav
Speech recognition complete.
Recognized Text (pa): ਲੋਨ ਬਾਰੇ ਦੱਸੋ
Recognized text stored in 'user_text_input': ਲੋਨ ਬਾਰੇ ਦੱਸੋ


In [ ]:
from googletrans import Translator

translator = Translator()

def translate_to_english(user_text_input: str, source_language_code: str) -> str:
    """
    Translates user input to English.
    Returns translated English text or error marker.
    """

    if not user_text_input:
        return "<EMPTY_INPUT>"

    if user_text_input.startswith("<Error:") or \
       user_text_input in [
           "<Could not understand audio>",
           "<Error with the speech recognition service>"
       ]:
        return user_text_input  # propagate error safely

    try:
        translated = translator.translate(
            user_text_input,
            src=source_language_code,
            dest="en"
        )
        return translated.text

    except Exception as e:
        print(f"Translation error: {e}")
        return user_text_input  # fallback to original text


In [ ]:
english_text = translate_to_english(user_text_input, source_language_code)

print("Text sent for intent detection:", english_text)


Text sent for intent detection: Tell about the loan


#Step1 = Intent Identification

We use a priority-based keyword intent detector on English-normalized text to route users into predefined backend flows.

#Define Intent Keywords

In [ ]:
INTENT_KEYWORDS = {
    "SCAM_CHECK": [
        "scam", "fraud", "otp", "suspicious", "fake", "message", "link",
        "loan call", "unknown call"
    ],

    "SCHEME_ELIGIBILITY": [
        "eligible", "eligibility", "am i eligible", "can i get",
        "qualify", "will i get"
    ],

    "SCHEME_INFO": [
        "scheme", "government scheme", "pm kisan", "kcc", "mudra", "yojana"
    ],

    "LOAN_EDUCATION": [
        "loan", "emi", "interest", "repayment", "installment"
    ],

    "APPLICATION_PROCESS": [
        "apply", "application", "how to apply", "where to apply", "process"
    ],

    "DOCUMENT_HELP": [
        "document", "documents", "aadhaar", "pan", "passbook", "proof"
    ],

    "USER_PROFILE": [
        "farmer", "shopkeeper", "business", "salaried", "job", "employment"
    ],

    "GREETING_HELP": [
        "hello", "hi", "help", "start", "explain", "tell me"
    ]
}


# Intent Priority Order

This is critical for speech input.

In [ ]:
INTENT_PRIORITY = [
    "SCAM_CHECK",
    "SCHEME_ELIGIBILITY",
    "SCHEME_INFO",
    "LOAN_EDUCATION",
    "APPLICATION_PROCESS",
    "DOCUMENT_HELP",
    "USER_PROFILE",
    "GREETING_HELP"
]


#Intent Detection Function

In [ ]:
# def detect_intent(user_text: str) -> str:
#     """
#     Detect intent from English-normalized user text.
#     Returns intent name as string.
#     """

#     if not user_text:
#         return "GREETING_HELP"

#     text = user_text.lower()

#     for intent in INTENT_PRIORITY:
#         keywords = INTENT_KEYWORDS[intent]
#         for keyword in keywords:
#             if keyword in text:
#                 return intent

#     return "GREETING_HELP"


In [ ]:
# # user_input = "Am I eligible for PM Kisan scheme?"
# user_input = "What is EMI?"

# # user_text = translated_english_text in our case.

# intent = detect_intent(user_input)

# print(intent)



LOAN_EDUCATION


#Regex Based Detection

Why this is better than plain keywords

Uses word boundaries to avoid false hits

Handles phrases (“how to apply”, “am I eligible”)

More STT-noise tolerant

Still explainable and auditable (no AI guesswork)

In [ ]:
import re
from typing import Dict, List

INTENT_PATTERNS: Dict[str, List[re.Pattern]] = {
    "SCAM_CHECK": [
        re.compile(r"\b(scam|fraud|fake|phishing)\b"),
        re.compile(r"\b(otp)\b"),
        re.compile(r"\b(suspicious|unknown)\s+(call|message|link)\b"),
        re.compile(r"\b(loan)\s+(call|app)\b"),
    ],

    "SCHEME_ELIGIBILITY": [
        re.compile(r"\b(am\s+i\s+eligible|eligibility|qualify)\b"),
        re.compile(r"\b(can\s+i\s+get|will\s+i\s+get)\b"),
    ],

    "SCHEME_INFO": [
        re.compile(r"\b(government\s+scheme|scheme|yojana)\b"),
        re.compile(r"\b(pm\s*kisan|kcc|mudra)\b"),
    ],

    "LOAN_EDUCATION": [
        re.compile(r"\b(loan)\b"),
        re.compile(r"\b(emi|interest|repayment|installment)\b"),
    ],

    "APPLICATION_PROCESS": [
        re.compile(r"\b(apply|application|process)\b"),
        re.compile(r"\b(how\s+to\s+apply|where\s+to\s+apply)\b"),
    ],

    "DOCUMENT_HELP": [
        re.compile(r"\b(document|documents)\b"),
        re.compile(r"\b(aadhaar|pan|passbook|proof)\b"),
    ],

    "USER_PROFILE": [
        re.compile(r"\b(farmer|agriculture)\b"),
        re.compile(r"\b(shopkeeper|business)\b"),
        re.compile(r"\b(salaried|job|employment)\b"),
    ],

    "GREETING_HELP": [
        re.compile(r"\b(hello|hi|help|start)\b"),
        re.compile(r"\b(explain|tell\s+me)\b"),
    ],
}


In [ ]:
INTENT_PRIORITY = [
    "SCAM_CHECK",
    "SCHEME_ELIGIBILITY",
    "SCHEME_INFO",
    "LOAN_EDUCATION",
    "APPLICATION_PROCESS",
    "DOCUMENT_HELP",
    "USER_PROFILE",
    "GREETING_HELP",
]

def detect_intent_regex(english_text: str) -> str:
    if not english_text:
        return "GREETING_HELP"

    text = english_text.lower().strip()

    for intent in INTENT_PRIORITY:
        for pattern in INTENT_PATTERNS[intent]:
            if pattern.search(text):
                return intent

    return "GREETING_HELP"


In [ ]:
# tests = [
#     "I received a suspicious loan call asking for OTP",
#     "Am I eligible for PM Kisan?",
#     "What documents are required?",
#     "How to apply for Mudra loan?",
#     "Hello, I need help"
# ]

# for t in tests:
#     print(t, "->", detect_intent_regex(t))

intent = detect_intent_regex(english_text)

print("Detected intent:", intent)

Detected intent: LOAN_EDUCATION


#Step2 = Bind Intent with the Flow
After intent detection, we deterministically bind the intent to a predefined flow and store it in the session to control all subsequent questions and logic.

#Define Flow Constants

In [ ]:
# flows.py

SYSTEM_START_FLOW = "SYSTEM_START_FLOW"
PROFILE_FLOW = "PROFILE_FLOW"
SCHEME_INFO_FLOW = "SCHEME_INFO_FLOW"
ELIGIBILITY_FLOW = "ELIGIBILITY_FLOW"
LOAN_EDU_FLOW = "LOAN_EDU_FLOW"
APPLICATION_FLOW = "APPLICATION_FLOW"
DOCUMENT_FLOW = "DOCUMENT_FLOW"
SCAM_FLOW = "SCAM_FLOW"


#Intent → Flow Mapping

In [ ]:
# intent_flow_map.py

INTENT_FLOW_MAP = {
    "GREETING_HELP": SYSTEM_START_FLOW,
    "USER_PROFILE": PROFILE_FLOW,
    "SCHEME_INFO": SCHEME_INFO_FLOW,
    "SCHEME_ELIGIBILITY": ELIGIBILITY_FLOW,
    "LOAN_EDUCATION": LOAN_EDU_FLOW,
    "APPLICATION_PROCESS": APPLICATION_FLOW,
    "DOCUMENT_HELP": DOCUMENT_FLOW,
    "SCAM_CHECK": SCAM_FLOW,
}


#Session Model (Minimal Example)

In [ ]:
# session.py

class Session:
    def __init__(self, session_id: str):
        self.session_id = session_id
        self.intent = None
        self.current_flow = None
        self.current_step = None

    def save(self):
        # Replace this with DB save logic
        pass


#Bind Intent to Flow (Core Function)

In [ ]:
# flow_binder.py

def bind_intent_to_flow(session, intent: str) -> str:
    """
    Binds detected intent to a predefined flow.
    Updates session state.
    """

    if intent not in INTENT_FLOW_MAP:
        # Safe fallback
        intent = "GREETING_HELP"

    flow = INTENT_FLOW_MAP[intent]

    # Update session
    session.intent = intent
    session.current_flow = flow
    session.current_step = 0  # always start at step 0

    session.save()

    return flow


In [ ]:
# from session import Session
# from intent_detector import detect_intent_regex
# from flow_binder import bind_intent_to_flow

# Assume this text is already translated to English
# user_text = "Am I eligible for PM Kisan?"

session = Session(session_id="abc123")

intent = detect_intent_regex(english_text)
flow = bind_intent_to_flow(session, intent)

print("Detected intent:", intent)
print("Bound flow:", flow)


Detected intent: LOAN_EDUCATION
Bound flow: LOAN_EDU_FLOW


#Step3- Define All Flow Question Sets

In [ ]:
FLOWS = {

    "SYSTEM_START_FLOW": [
        {
            "key": "start",
            "question": "How can I help you today?",
            "required": True
        }
    ],

    "PROFILE_FLOW": [
        {
            "key": "occupation",
            "question": "What is your occupation? (Farmer / Shopkeeper / Salaried / Other)",
            "required": True
        }
    ],

    "SCHEME_INFO_FLOW": [
        {
            "key": "scheme_name",
            "question": "Which government scheme do you want to know about?",
            "required": True
        }
    ],

    "ELIGIBILITY_FLOW": [
        {
            "key": "occupation",
            "question": "What is your occupation?",
            "required": True
        },
        {
            "key": "land_owned",
            "question": "Do you own agricultural land? (yes/no)",
            "required": True
        },
        {
            "key": "income_range",
            "question": "What is your annual income range?",
            "required": False
        },
        {
            "key": "bank_account",
            "question": "Do you have a bank account? (yes/no)",
            "required": True
        }
    ],

    "LOAN_EDU_FLOW": [
        {
            "key": "loan_topic",
            "question": "What do you want to understand? (EMI / Interest / Repayment)",
            "required": True
        }
    ],

    "APPLICATION_FLOW": [
        {
            "key": "application_target",
            "question": "Which scheme or loan do you want to apply for?",
            "required": True
        }
    ],

    "DOCUMENT_FLOW": [
        {
            "key": "document_for",
            "question": "Documents are needed for which scheme or loan?",
            "required": True
        }
    ],

    "SCAM_FLOW": [
        {
            "key": "scam_description",
            "question": "Please describe the suspicious call, message, or app.",
            "required": True
        }
    ]
}


In [ ]:
class Session:
    def __init__(self, session_id):
        self.session_id = session_id
        self.intent = None
        self.current_flow = None
        self.current_step = 0
        self.answers = {}
    def save(self):
        pass  # placeholder for DB persistence later


In [ ]:
def get_next_question(session: Session):
    flow = FLOWS.get(session.current_flow, [])

    while session.current_step < len(flow):
        q = flow[session.current_step]
        if q["key"] not in session.answers:
            return q
        session.current_step += 1

    return None


In [ ]:
def store_answer(session: Session, answer: str):
    flow = FLOWS[session.current_flow]
    q = flow[session.current_step]

    session.answers[q["key"]] = answer
    session.current_step += 1


In [ ]:
# Create session
session = Session("demo001")

# Simulate detected intent
detected_intent = intent  # try others too

bind_intent_to_flow(session, detected_intent)

print("Intent:", session.intent)
print("Flow:", session.current_flow)

while True:
    q = get_next_question(session)
    if q is None:
        print("\nAll questions completed.")
        print("Collected Data:", session.answers)
        break

    print("\nSYSTEM:", q["question"])
    user_input = input("USER: ")
    store_answer(session, user_input)


Intent: LOAN_EDUCATION
Flow: LOAN_EDU_FLOW

SYSTEM: What do you want to understand? (EMI / Interest / Repayment)
USER: Repayment

All questions completed.
Collected Data: {'loan_topic': 'Repayment'}


#STEP 3 — Backend Rule Engine (Eligibility Logic)
This is only for scheme eligibility , every other intent will have its own

In [ ]:
def normalize_answers(data: dict):
    normalized = {}
    for k, v in data.items():
        if isinstance(v, str):
            normalized[k] = v.strip().lower()
        else:
            normalized[k] = v
    return normalized

In [ ]:
SCHEME_RULES = [ # data to be collected.%
    {
        "name": "PM-Kisan",
        "conditions": {
            "occupation": "farmer",
            "land_owned": "yes",
            "bank_account": "yes"
        }
    },
    {
        "name": "Kisan Credit Card (KCC)",
        "conditions": {
            "occupation": "farmer",
            "bank_account": "yes"
        }
    },
    {
        "name": "Mudra Loan",
        "conditions": {
            "occupation": ["shopkeeper", "business"],
            "bank_account": "yes"
        }
    },
    {
        "name": "PM-SVANidhi",
        "conditions": {
            "occupation": "shopkeeper",
            "bank_account": "yes"
        }
    },
    {
        "name": "Jan Dhan Account",
        "conditions": {
            "bank_account": "no"
        }
    }
]

In [ ]:
def check_conditions(answers: dict, conditions: dict) -> bool:
    for key, expected in conditions.items():
        actual = answers.get(key)

        if isinstance(expected, list):
            if actual not in expected:
                return False
        else:
            if actual != expected:
                return False

    return True


#PROCESSORS (One per Intent)

##SCHEME_ELIGIBILITY Processor

In [ ]:
# def eligibility_rules(answers: dict):
#     """
#     Apply deterministic eligibility rules.
#     """

#     answers = normalize_answers(answers)

#     eligible_schemes = []

#     occupation = answers.get("occupation", "")
#     land_owned = answers.get("land_owned", "")
#     bank_account = answers.get("bank_account", "")

#     # ---- PM-Kisan ----
#     if occupation == "farmer" and land_owned == "yes" and bank_account == "yes":
#         eligible_schemes.append("PM-Kisan")

#     # ---- Kisan Credit Card (KCC) ----
#     if occupation == "farmer" and bank_account == "yes":
#         eligible_schemes.append("Kisan Credit Card (KCC)")

#     # ---- Mudra Loan ----
#     if occupation in ["shopkeeper", "business"] and bank_account == "yes":
#         eligible_schemes.append("Mudra Loan")

#     return {
#         "eligible_schemes": eligible_schemes,
#         "normalized_answers": answers
#     }

def process_scheme_eligibility(answers: dict):
    answers = normalize_answers(answers)
    eligible = []

    for scheme in SCHEME_RULES:
        if check_conditions(answers, scheme["conditions"]):
            eligible.append(scheme["name"])

    return {
        "type": "ELIGIBILITY_RESULT",
        "eligible_schemes": eligible,
        "answers": answers
    }



#SCHEME_INFO Processor

In [ ]:
def process_scheme_info(answers: dict):
    scheme = answers.get("scheme_name", "unknown")

    return {
        "type": "SCHEME_INFO",
        "scheme": scheme,
        "note": "Explain scheme details via LLM"
    }

#LOAN_EDUCATION Processor

In [ ]:
def process_loan_education(answers: dict):
    topic = answers.get("loan_topic", "loan")

    return {
        "type": "LOAN_EDUCATION",
        "topic": topic,
        "note": "Educational explanation only"
    }

#APPLICATION_PROCESS Processor

In [ ]:
def process_application(answers: dict):
    target = answers.get("application_target", "scheme")

    return {
        "type": "APPLICATION_GUIDE",
        "target": target,
        "steps": [
            "Visit nearest bank or CSC center",
            "Carry required documents",
            "Fill application form"
        ]
    }

#DOCUMENT_HELP Processor

In [ ]:
def process_documents(answers: dict):
    target = answers.get("document_for", "scheme")

    return {
        "type": "DOCUMENT_LIST",
        "target": target,
        "documents": [
            "Aadhaar Card",
            "Bank Passbook",
            "Address Proof"
        ]
    }


#SCAM_CHECK Processor

In [ ]:
def process_scam_check(answers: dict):
    description = answers.get("scam_description", "")

    return {
        "type": "SCAM_WARNING",
        "message": "Do not share OTP, PIN, or click unknown links.",
        "action": "Report to bank or call 1930 cybercrime helpline"
    }


#GREETING / SYSTEM Processor

In [ ]:
def process_greeting():
    return {
        "type": "GREETING",
        "message": "How can I help you today?"
    }


In [ ]:
# def process_flow_completion(session: Session):
#     """
#     Called when all flow questions are answered.
#     Applies backend rules and returns structured result.
#     """

#     if session.current_flow == "ELIGIBILITY_FLOW":
#         return eligibility_rules(session.answers)

#     # Other flows can have their own processors later
#     return {}


In [ ]:
# After all questions are answered
# result = process_flow_completion(session)

# print("\n=== BACKEND DECISION (NO LLM) ===")
# print(result)



=== BACKEND DECISION (NO LLM) ===
{'eligible_schemes': [], 'normalized_answers': {'occupation': 'teacher', 'land_owned': 'no', 'income_range': '5 lakhs', 'bank_account': 'yes'}}


In [ ]:
def backend_rule_engine(session: Session):
    flow = session.current_flow
    answers = session.answers

    if flow == "ELIGIBILITY_FLOW":
        return process_scheme_eligibility(answers)

    elif flow == "SCHEME_INFO_FLOW":
        return process_scheme_info(answers)

    elif flow == "LOAN_EDU_FLOW":
        return process_loan_education(answers)

    elif flow == "APPLICATION_FLOW":
        return process_application(answers)

    elif flow == "DOCUMENT_FLOW":
        return process_documents(answers)

    elif flow == "SCAM_FLOW":
        return process_scam_check(answers)

    elif flow == "SYSTEM_START_FLOW":
        return process_greeting()

    else:
        return {
            "type": "UNKNOWN",
            "message": "Unable to process request"
        }


In [ ]:
if q is None:
    print("\nAll questions completed.")
    print("Collected Data:", session.answers)

    result = backend_rule_engine(session)

    print("\n=== BACKEND RULE ENGINE OUTPUT ===")
    print(result)




All questions completed.
Collected Data: {'loan_topic': 'Repayment'}

=== BACKEND RULE ENGINE OUTPUT ===
{'type': 'LOAN_EDUCATION', 'topic': 'Repayment', 'note': 'Educational explanation only'}


“Follow-up questions in Grameen-AI are backend-controlled system prompts used only to guide next actions. The LLM never asks questions; it only explains results.”

Final mental model (keep this)

Flows ask questions

Rules decide

LLM explains

Backend offers next step

User chooses

In [ ]:
!pip install -q google-generativeai

In [ ]:
import google.generativeai as genai
from google.colab import userdata

# Install required Python libraries
get_ipython().system('pip install SpeechRecognition pydub googletrans==4.0.0-rc1')
print("SpeechRecognition, pydub, and googletrans libraries installed.")

# Install ffmpeg system-level dependency
get_ipython().system('apt-get install -y ffmpeg')
print("ffmpeg installed.")

# Initialize gemini_model to None
gemini_model = None

try:
    # Access GOOGLE_API_KEY
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')

    # Configure genai library
    genai.configure(api_key=GOOGLE_API_KEY)

    # Initialize the gemini_model
    gemini_model = genai.GenerativeModel('gemini-flash-latest')
    print("Gemini model configured and initialized successfully.")

except userdata.SecretNotFoundError:
    print("ERROR: GOOGLE_API_KEY not found in Colab secrets.")
    print("Please ensure you have set your API key in Colab's secrets manager (🔑 icon on the left panel) and named it 'GOOGLE_API_KEY'.")
    print("Skipping Gemini model initialization and response generation.")
except Exception as e:
    print(f"An unexpected error occurred during Gemini API configuration: {e}")

SpeechRecognition, pydub, and googletrans libraries installed.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 1 not upgraded.
ffmpeg installed.
Gemini model configured and initialized successfully.


In [ ]:
def build_explanation_prompt(backend_result: dict, language: str = "English"):

    result_type = backend_result.get("type")

    if result_type == "ELIGIBILITY_RESULT":
        schemes = backend_result["eligible_schemes"]

        if schemes:
            decision_text = "The user is eligible for: " + ", ".join(schemes)
        else:
            decision_text = "The user is not eligible for any scheme."

        return f"""
You are a financial assistant for rural users in India of Punjab Region who do not understand very critical terms.

RULES:
- Do NOT decide eligibility
- Do NOT add new schemes
- Do NOT ask questions
- ONLY explain the backend decision

Backend decision:
{decision_text}

Explain clearly in {language} using very simple words to the rural people of Punjab Region.
"""

    elif result_type == "SCHEME_INFO":
        return f"""
Explain the government scheme mentioned below in {language}.
Use simple words and practical examples based on Punjab region, India.
Do not give advice or eligibility decisions but Give eligibility info.

Scheme:
{backend_result.get("scheme")}
"""

    elif result_type == "LOAN_EDUCATION":
        return f"""
Explain the following financial concept in {language}:
{backend_result.get("topic")} with respect to Punjab region, India.
Use simple words and practical examples. Also explain in brief do not go too deep.

Rules:
- Educational explanation only
- No recommendations
- No approvals
"""

    elif result_type == "SCAM_WARNING":
        return f"""
Explain the safety warning below in {language}.
Be firm and clear.
Do not ask questions.

Warning:
{backend_result.get("message")}
"""

    else:
        return f"Explain the following information in simple {language}."



In [ ]:
def explain_with_gemini(backend_result: dict, language: str = "English"):
    prompt = build_explanation_prompt(backend_result, language)

    response = gemini_model.generate_content(prompt)

    return response.text.strip()


In [ ]:
backend_result = backend_rule_engine(session)

print("\n=== BACKEND RESULT (NO AI) ===")
print(backend_result)

# 🔵 LLM EXPLANATION (ONLY HERE)
explanation = explain_with_gemini(
    backend_result,
    language="English"  # later: Hindi / Punjabi / Telugu
)

print("\n=== GEMINI EXPLANATION ===")
print(explanation)



=== BACKEND RESULT (NO AI) ===
{'type': 'LOAN_EDUCATION', 'topic': 'Repayment', 'note': 'Educational explanation only'}

=== GEMINI EXPLANATION ===
This is an explanation of the concept of **Repayment** specifically focusing on its context within the **Punjab region, India**.

---

## Financial Concept: Repayment (Punjab Context)

**Repayment** is a simple financial concept: it means **paying back money that you borrowed**.

If you take a loan from a bank, a moneylender, or even a friend, the act of giving that money back (usually in installments with interest) is called repayment.

### 1. Simple Explanation

| Term | What it Means |
| :--- | :--- |
| **Loan** | Money borrowed. |
| **Borrower** | The person who takes the money (e.g., a farmer). |
| **Lender** | The institution or person who gives the money (e.g., a bank). |
| **Repayment** | The process of the borrower giving the money back to the lender. |

---

### 2. Repayment in the Punjab Context: Practical Examples

The concept 

#Translation into Punjabi

In [ ]:
source_language_code = "en" # for english
target_language_code = "pa" # punjabi


In [ ]:
from googletrans import Translator

translator = Translator()

def translate_to_english(user_text_input: str, source_language_code: str) -> str:
    """
    Translates user input to English.
    Returns translated English text or error marker.
    """

    if not user_text_input:
        return "<EMPTY_INPUT>"

    if user_text_input.startswith("<Error:") or \
       user_text_input in [
           "<Could not understand audio>",
           "<Error with the speech recognition service>"
       ]:
        return user_text_input  # propagate error safely

    try:
        translated = translator.translate(
            user_text_input,
            src=source_language_code,
            dest="pa"
        )
        return translated.text

    except Exception as e:
        print(f"Translation error: {e}")
        return user_text_input  # fallback to original text

In [ ]:
punjabi_translated = translate_to_english(explanation, source_language_code)
print("\n=== PUNJABI TRANSLATION ===")
print(punjabi_translated)


=== PUNJABI TRANSLATION ===
ਇਹ **ਮੁੜਭੁਗਤਾਨ** ਦੀ ਧਾਰਨਾ ਦੀ ਵਿਆਖਿਆ ਹੈ ਜੋ ਵਿਸ਼ੇਸ਼ ਤੌਰ 'ਤੇ **ਪੰਜਾਬ ਖੇਤਰ, ਭਾਰਤ** ਦੇ ਅੰਦਰ ਇਸ ਦੇ ਸੰਦਰਭ 'ਤੇ ਕੇਂਦਰਿਤ ਹੈ।

---

## ਵਿੱਤੀ ਸੰਕਲਪ: ਮੁੜ ਅਦਾਇਗੀ (ਪੰਜਾਬ ਸੰਦਰਭ)

**ਮੁੜ-ਭੁਗਤਾਨ** ਇੱਕ ਸਧਾਰਨ ਵਿੱਤੀ ਸੰਕਲਪ ਹੈ: ਇਸਦਾ ਮਤਲਬ ਹੈ **ਤੁਹਾਡੇ ਵੱਲੋਂ ਉਧਾਰ ਲਏ ਪੈਸੇ ਦਾ ਭੁਗਤਾਨ ਕਰਨਾ**।

ਜੇਕਰ ਤੁਸੀਂ ਕਿਸੇ ਬੈਂਕ, ਸ਼ਾਹੂਕਾਰ, ਜਾਂ ਇੱਥੋਂ ਤੱਕ ਕਿ ਕਿਸੇ ਦੋਸਤ ਤੋਂ ਕਰਜ਼ਾ ਲੈਂਦੇ ਹੋ, ਤਾਂ ਉਸ ਪੈਸੇ ਨੂੰ ਵਾਪਸ ਦੇਣ ਦੀ ਕਾਰਵਾਈ (ਆਮ ਤੌਰ 'ਤੇ ਵਿਆਜ ਸਮੇਤ ਕਿਸ਼ਤਾਂ ਵਿੱਚ) ਨੂੰ ਮੁੜ ਭੁਗਤਾਨ ਕਿਹਾ ਜਾਂਦਾ ਹੈ।

### 1. ਸਰਲ ਵਿਆਖਿਆ

|ਮਿਆਦ |ਇਸਦਾ ਕੀ ਮਤਲਬ ਹੈ |
|:--- |:--- |
|**ਕਰਜ਼ਾ** |ਪੈਸੇ ਉਧਾਰ ਲਏ।|
|**ਕਰਜ਼ਦਾਰ** |ਉਹ ਵਿਅਕਤੀ ਜੋ ਪੈਸੇ ਲੈਂਦਾ ਹੈ (ਉਦਾਹਰਨ ਲਈ, ਇੱਕ ਕਿਸਾਨ)।|
|**ਉਧਾਰ ਦੇਣ ਵਾਲਾ** |ਸੰਸਥਾ ਜਾਂ ਵਿਅਕਤੀ ਜੋ ਪੈਸਾ ਦਿੰਦਾ ਹੈ (ਉਦਾਹਰਨ ਲਈ, ਇੱਕ ਬੈਂਕ)।|
|**ਮੁੜ ਅਦਾਇਗੀ** |ਉਧਾਰ ਲੈਣ ਵਾਲੇ ਦੁਆਰਾ ਉਧਾਰ ਦੇਣ ਵਾਲੇ ਨੂੰ ਪੈਸੇ ਵਾਪਸ ਦੇਣ ਦੀ ਪ੍ਰਕਿਰਿਆ।|

---

### 2. ਪੰਜਾਬ ਸੰਦਰਭ ਵਿੱਚ ਮੁੜ ਅਦਾਇਗੀ: ਵਿਹਾਰਕ ਉਦਾਹਰਨਾਂ

ਪੰਜਾਬ ਵਿੱਚ ਮੁੜ ਅਦਾਇਗੀ ਦੀ ਧਾਰਨਾ ਖਾਸ ਤੌਰ 'ਤੇ ਮਹੱਤਵਪੂਰਨ ਹੈ ਕਿਉਂਕਿ ਆਰਥਿਕਤਾ **ਖੇਤੀਬਾੜੀ** ਅਤੇ **ਛੋਟੇ ਕਾਰੋਬਾਰਾਂ (SMEs)** 'ਤੇ ਬਹੁਤ ਜ਼ਿਆਦਾ ਨਿਰਭਰ ਕਰਦੀ ਹੈ, ਜਿਨ੍ਹਾਂ ਦੋ

In [ ]:
def clean_text(text: str) -> str:
    if not text:
        return text

    # Convert escaped newlines to real newlines
    text = text.replace("\\n", "\n")

    # Optional: remove excessive backslashes
    text = text.replace("\\", "")

    return text

In [ ]:
import re

def remove_markdown(text: str) -> str:
    # Remove markdown headers
    text = re.sub(r"#+\s*", "", text)

    # Remove bold/italic markers
    text = re.sub(r"\*\*", "", text)
    text = re.sub(r"\*", "", text)

    return text

cleaned = clean_text(punjabi_translated)
cleaned = remove_markdown(cleaned)

print(cleaned)


ਇਹ ਮੁੜਭੁਗਤਾਨ ਦੀ ਧਾਰਨਾ ਦੀ ਵਿਆਖਿਆ ਹੈ ਜੋ ਵਿਸ਼ੇਸ਼ ਤੌਰ 'ਤੇ ਪੰਜਾਬ ਖੇਤਰ, ਭਾਰਤ ਦੇ ਅੰਦਰ ਇਸ ਦੇ ਸੰਦਰਭ 'ਤੇ ਕੇਂਦਰਿਤ ਹੈ।

---

ਵਿੱਤੀ ਸੰਕਲਪ: ਮੁੜ ਅਦਾਇਗੀ (ਪੰਜਾਬ ਸੰਦਰਭ)

ਮੁੜ-ਭੁਗਤਾਨ ਇੱਕ ਸਧਾਰਨ ਵਿੱਤੀ ਸੰਕਲਪ ਹੈ: ਇਸਦਾ ਮਤਲਬ ਹੈ ਤੁਹਾਡੇ ਵੱਲੋਂ ਉਧਾਰ ਲਏ ਪੈਸੇ ਦਾ ਭੁਗਤਾਨ ਕਰਨਾ।

ਜੇਕਰ ਤੁਸੀਂ ਕਿਸੇ ਬੈਂਕ, ਸ਼ਾਹੂਕਾਰ, ਜਾਂ ਇੱਥੋਂ ਤੱਕ ਕਿ ਕਿਸੇ ਦੋਸਤ ਤੋਂ ਕਰਜ਼ਾ ਲੈਂਦੇ ਹੋ, ਤਾਂ ਉਸ ਪੈਸੇ ਨੂੰ ਵਾਪਸ ਦੇਣ ਦੀ ਕਾਰਵਾਈ (ਆਮ ਤੌਰ 'ਤੇ ਵਿਆਜ ਸਮੇਤ ਕਿਸ਼ਤਾਂ ਵਿੱਚ) ਨੂੰ ਮੁੜ ਭੁਗਤਾਨ ਕਿਹਾ ਜਾਂਦਾ ਹੈ।

1. ਸਰਲ ਵਿਆਖਿਆ

|ਮਿਆਦ |ਇਸਦਾ ਕੀ ਮਤਲਬ ਹੈ |
|:--- |:--- |
|ਕਰਜ਼ਾ |ਪੈਸੇ ਉਧਾਰ ਲਏ।|
|ਕਰਜ਼ਦਾਰ |ਉਹ ਵਿਅਕਤੀ ਜੋ ਪੈਸੇ ਲੈਂਦਾ ਹੈ (ਉਦਾਹਰਨ ਲਈ, ਇੱਕ ਕਿਸਾਨ)।|
|ਉਧਾਰ ਦੇਣ ਵਾਲਾ |ਸੰਸਥਾ ਜਾਂ ਵਿਅਕਤੀ ਜੋ ਪੈਸਾ ਦਿੰਦਾ ਹੈ (ਉਦਾਹਰਨ ਲਈ, ਇੱਕ ਬੈਂਕ)।|
|ਮੁੜ ਅਦਾਇਗੀ |ਉਧਾਰ ਲੈਣ ਵਾਲੇ ਦੁਆਰਾ ਉਧਾਰ ਦੇਣ ਵਾਲੇ ਨੂੰ ਪੈਸੇ ਵਾਪਸ ਦੇਣ ਦੀ ਪ੍ਰਕਿਰਿਆ।|

---

2. ਪੰਜਾਬ ਸੰਦਰਭ ਵਿੱਚ ਮੁੜ ਅਦਾਇਗੀ: ਵਿਹਾਰਕ ਉਦਾਹਰਨਾਂ

ਪੰਜਾਬ ਵਿੱਚ ਮੁੜ ਅਦਾਇਗੀ ਦੀ ਧਾਰਨਾ ਖਾਸ ਤੌਰ 'ਤੇ ਮਹੱਤਵਪੂਰਨ ਹੈ ਕਿਉਂਕਿ ਆਰਥਿਕਤਾ ਖੇਤੀਬਾੜੀ ਅਤੇ ਛੋਟੇ ਕਾਰੋਬਾਰਾਂ (SMEs) 'ਤੇ ਬਹੁਤ ਜ਼ਿਆਦਾ ਨਿਰਭਰ ਕਰਦੀ ਹੈ, ਜਿਨ੍ਹਾਂ ਦੋਵਾਂ ਨੂੰ ਅਕਸਰ ਕਰਜ਼ੇ ਦੀ ਲੋੜ ਹੁੰਦੀ ਹੈ।

ਇੱਥੇ ਮੁੱਖ ਖੇਤਰ ਹਨ ਜਿੱਥੇ ਮੁੜ ਅਦਾਇਗੀ ਹੁੰਦੀ ਹੈ